# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIRˆ²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print out dataset name and description
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets by `@id`
print("Available Record Sets (by @id):")
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in the metadata.\n")
else:
    for rs in record_sets:
        print(f"- @id: {rs.id}, name: {rs.name}")

# For this dataset, we access top-level record sets from the metadata
# Let's programmatically get their IDs and display their fields
record_set_ids = [rs.id for rs in dataset.metadata.record_sets] if record_sets else []
for rs in dataset.metadata.record_sets:
    print(f"\nRecord Set: {rs.name} (@id: {rs.id})")
    print("  Fields:")
    if hasattr(rs, "fields"):
        for field in rs.fields:
            print(f"    - @id: {field.id}, name: {field.name}, data type: {field.data_type}")
    else:
        print("    No fields available.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to extract data from record sets into DataFrames

if not record_set_ids:
    print("No record sets to extract from.")
else:
    dataframes = {}
    for record_set_id in record_set_ids:
        print(f"Loading records for RecordSet @id: {record_set_id}")
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            if not df.empty:
                print(f"Fields/Columns: {df.columns.tolist()}")
                display(df.head())
            else:
                print("No records found.")
        except Exception as e:
            print(f"Error loading records: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers, transforming data distributions, or grouping data by key attributes to prepare for analysis.

In [ ]:
# Example: Filter and normalize using one recordset (if available)
import numpy as np

if record_set_ids:
    # Pick the first record set with data
    main_record_set_id = None
    for rs_id in record_set_ids:
        if rs_id in dataframes and not dataframes[rs_id].empty:
            main_record_set_id = rs_id
            break

    if main_record_set_id is not None:
        df = dataframes[main_record_set_id]
        print(f"Using RecordSet: {main_record_set_id}")

        # Automatically select a numeric field (float/int)
        numeric_field = None
        for col in df.columns:
            # Try to infer numeric column
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
        if numeric_field is None:
            # Try to force convert columns to numeric
            for col in df.columns:
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    if pd.api.types.is_numeric_dtype(df[col]) and df[col].count() > 0:
                        numeric_field = col
                        break
                except Exception:
                    continue

        if numeric_field is not None:
            print(f"Selected numeric field for analysis: {numeric_field}")
            # Filter: keep only values above a certain threshold, e.g., mean
            threshold = df[numeric_field].mean()
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
            display(filtered_df.head())

            # Normalize the numeric field (z-score)
            filtered_df[f"{numeric_field}_normalized"] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            )
            print(f"\nNormalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Try grouping by a string/categorical field (if any)
            group_field = None
            for col in filtered_df.columns:
                if col != numeric_field and pd.api.types.is_string_dtype(filtered_df[col]):
                    group_field = col
                    break
            if group_field is not None:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
                print(f"\nGrouped data by '{group_field}' (mean of {numeric_field}):")
                display(grouped_df.head())
            else:
                print("No suitable group field found for grouping.")
        else:
            print("No numeric fields found for EDA.")
    else:
        print("No record set with data available for EDA.")
else:
    print("No record sets available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and 'main_record_set_id' in locals() and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    if 'numeric_field' in locals() and numeric_field in df.columns:
        plt.figure(figsize=(8,5))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field} in main record set")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()
    else:
        print("No numeric field to visualize.")
else:
    print("No main record set found for visualization.")

## 6. Conclusion
In this notebook, we used `mlcroissant` to:
- Load the [FAIRˆ² Mass Spectrometry EVs Dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa)
- Explore its record sets and fields by `@id`
- Load records into Pandas DataFrames
- Perform basic EDA, including filtering, normalization, and grouping
- Visualize numeric data distributions

This dataset can further be analyzed for biological patterns, batch effects, or used as input for modeling as appropriate.

For more advanced dataset exploration, see the [mlcroissant documentation](https://mlcommons.org/croissant) and the full dataset schema.